# Debugging, Monitoring & Tool Calling with LangChain and LangGraph

Yeh notebook demonstrate karta hai ki kaise hum LangChain mein models initialize karte hain, unka execution monitor karte hain, LangSmith tracing use karte hain, aur LangGraph ke through tool-augmented agent workflows build karte hain.

### Environment Setup & Config
- **Purpose**: `.env` file se API keys (`GROQ_API_KEY`, `LANGCHAIN_API_KEY`) aur LangSmith tracing parameters load karna.
- **Variables**:
  - `LANGCHAIN_TRACING_V2`: Tracing ko enable/disable karta hai (`true`).
  - `LANGCHAIN_PROJECT`: LangSmith dashboard ke andar project ka naam set karta hai.
- **Explanation**: Tracing on hone ke baad har LLM call aur chain run automatically LangSmith dashboard par log hoti hai.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

print("Tracing Enabled:", os.getenv("LANGCHAIN_TRACING_V2"))
print("LangSmith Project:", os.getenv("LANGCHAIN_PROJECT"))
print("Groq API Key Configured:", bool(os.getenv("GROQ_API_KEY")))

### `init_chat_model` (Model Initialization)
- **Purpose**: LangChain ka unified function jo dynamically kisi bhi LLM provider (Groq, OpenAI, Anthropic, etc.) ka model initialize karta hai.
- **Parameters**:
  - `model` (`str`): Model ka identifier (e.g., `'openai/gpt-oss-20b'`).
  - `model_provider` (`str`): Underlying provider backend (e.g., `'groq'`).
- **Return Value**: An initialized Chat Model instance (e.g., `ChatGroq`).
- **Explanation**: Yeh hardcoded class imports (jaise `ChatGroq(...)`) ki jagah provider-agnostic initialization allow karta hai.

In [ ]:
from langchain.chat_models import init_chat_model

# Initialize Chat Model dynamically with Groq backend
model = init_chat_model(model='openai/gpt-oss-20b', model_provider='groq')
print("Initialized model:", model)

### Model Invocation & Response Metadata Inspection
- **Purpose**: Initialized model ko query bhejkar response, latency, token consumption aur execution metadata inspect karna.
- **Parameters**:
  - `input` (`str` ya `list[BaseMessage]`): Prompt message jo model ko pass hota hai.
- **Return Value**: `AIMessage` object jisme content, `response_metadata`, aur `usage_metadata` hote hain.
- **Explanation**: Debugging ke time token usage aur completion time verify karna zaroori hota hai.

In [ ]:
# Model call and metadata inspection
response = model.invoke("Explain debugging in distributed LLM applications in one crisp sentence.")

print("=== AI Response ===")
print(response.content)
print("\n=== Token Usage ===")
print(response.usage_metadata)
print("\n=== Response Metadata ===")
print(response.response_metadata)

### `MonitoringCallbackHandler` (Custom Callback Middleware)
- **Purpose**: LLM call ke lifecycle events ko intercept karke real-time logging aur debugging perform karna.
- **Methods**:
  - `on_llm_start`: Call start hone par prompt details capture karta hai.
  - `on_llm_end`: Response receive hone par latency aur output log karta hai.
  - `on_llm_error`: Agar execution fail hoti hai toh error stack trace capture karta hai.
- **Return Value**: None (side-effect logging).
- **Explanation**: Custom callbacks client-side debugging aur alert systems create karne ke liye use hote hain.

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler

class MonitoringCallbackHandler(BaseCallbackHandler):
    """Custom callback handler to monitor LLM execution steps."""
    def on_llm_start(self, serialized, prompts, **kwargs):
        print("🔍 [MONITORING] LLM Call Started...")
        print(f"   Prompt sent: {prompts[:1]}")

    def on_llm_end(self, response, **kwargs):
        print("✅ [MONITORING] LLM Call Finished Successfully!")

    def on_llm_error(self, error, **kwargs):
        print(f"❌ [MONITORING] LLM Error encountered: {error}")

# Invoke model with callback attached
callback_handler = MonitoringCallbackHandler()
monitored_res = model.invoke(
    "What is LangGraph?",
    config={"callbacks": [callback_handler]}
)
print("\nMonitored Content:", monitored_res.content)

### LangSmith Tracing with Custom Tags & Run Names
- **Purpose**: LangSmith dashboard par specific runs ko tag, metadata, aur meaningful run name ke saath filter karne ke liye configure karna.
- **Parameters in `config`**:
  - `run_name` (`str`): Custom readable run identifier.
  - `tags` (`list[str]`): Filtering labels (e.g., `["debugging", "groq"]`).
  - `metadata` (`dict`): Extra context (e.g., `user_id`, `environment`).
- **Return Value**: `AIMessage` containing execution output.
- **Explanation**: Production observability mein user sessions aur feature runs ko isolate karne ke liye tags aur metadata critical hote hain.

In [ ]:
# Invoke model with custom tracing metadata and tags
traced_response = model.invoke(
    "How does LangSmith observability help in production agent monitoring?",
    config={
        "run_name": "debug_monitoring_session_01",
        "tags": ["debugging", "monitoring", "groq", "gpt-oss-20b"],
        "metadata": {
            "environment": "development",
            "developer": "mohit_payasi"
        }
    }
)

print(traced_response.content)

### Tool Calling & Graph Imports
- **Purpose**: LangGraph mein tool calling aur message handling ke liye zaroori classes aur helper functions import karna.
- **Key Modules**:
  - `StateGraph`, `START`, `END`: Graph workflow aur state machine control.
  - `add_messages`: Message history ko overwrite karne ke bajaye append karne wala reducer.
  - `ToolNode`, `tools_condition`: Prebuilt tool execution node aur conditional routing helper.
  - `@tool`: Python functions ko LLM-callable tools mein convert karta hai.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.tools import Tool, tool
from langgraph.prebuilt import ToolNode, tools_condition
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage

### `state` Schema Definition (TypedDict with Reducer)
- **Purpose**: Graph execution ke dauran shared state aur conversation history hold karna.
- **Fields**:
  - `messages`: `Annotated[list[BaseMessage], add_messages]` - Har naya message state mein existing list ke end mein append hota hai.
- **Explanation**: `add_messages` reducer ensure karta hai ki purani messages retain rahein aur context loss na ho.

In [ ]:
class state(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

### `@tool` Definition & `ToolNode` Initialization
- **Function Name**: `add(a: float, b: float)`
- **Purpose**: Do numbers ko add karke unka mathematical sum return karna.
- **Parameters**:
  - `a` (`float`): First operand
  - `b` (`float`): Second operand
- **Return Value**: `float` (Sum of `a` and `b`)
- **`ToolNode`**: Prebuilt node jo LLM ke generated `tool_calls` ko receive karke actual Python function execute karta hai.

In [ ]:
@tool
def add(a: float, b: float):
    """add two numbers"""
    return a + b

tool_node = ToolNode([add])

### Tool Binding (`bind_tools`) & LLM Node
- **Function Name**: `call_llm_model(state)`
- **Purpose**: Model ko tools ke schema ke sath bind karna aur current message history pass karke response ya tool call generate karna.
- **Parameters**:
  - `state` (`state`): Current graph state containing conversation messages.
- **Return Value**: `dict` containing updated `messages` list with LLM's response.
- **Explanation**: `model.bind_tools([add])` model ko aware karta hai ki `add` tool available hai invocation ke liye.

In [ ]:
model_with_tool = model.bind_tools([add])

# Node definition for invoking model with tool binding
def call_llm_model(state: state):
    return {'messages': [model_with_tool.invoke(state['messages'])]}

### `StateGraph` Construction & Routing
- **Purpose**: State schema ke basis par workflow structure initialize karna, nodes add karna, aur conditional routing setup karna.
- **Components**:
  - Nodes: `'call_llm_model'`, `'tools'`
  - Conditional Edge: `tools_condition` route karta hai ki agar LLM ne tool call kiya hai toh `'tools'` par jaye, warna `END` par jaye.

In [ ]:
builder = StateGraph(state)
builder.add_node('call_llm_model', call_llm_model)
builder.add_node('tools', tool_node)

builder.add_edge(START, 'call_llm_model')
builder.add_conditional_edges('call_llm_model', tools_condition)
builder.add_edge('tools', 'call_llm_model')

graph = builder.compile()